### Create Silver Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS openalex_lakehouse.silver;

### Imports & Config

In [0]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType


CATALOG = "openalex_lakehouse"
SCHEMA = "silver"
TABLE = "works_clean"

SOURCE_TABLE = "openalex_lakehouse.bronze.works_raw"
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"


### Read bronze table

In [0]:
bronze_df = spark.table(SOURCE_TABLE)
bronze_df.display(2, truncate=True)

In [0]:
bronze_df.printSchema()

In [0]:
from pyspark.sql.functions import col, to_date, get_json_object

RAW_JSON_COL = "raw_json"

silver_df = bronze_df.select(
    get_json_object(RAW_JSON_COL, "$.id").alias("work_id"),
    get_json_object(RAW_JSON_COL, "$.title").alias("title"),
    get_json_object(RAW_JSON_COL, "$.publication_year").alias("publication_year"),
    get_json_object(RAW_JSON_COL, "$.publication_date").alias("publication_date"),
    get_json_object(RAW_JSON_COL, "$.cited_by_count").cast("int").alias("citation_count"),
    get_json_object(RAW_JSON_COL, "$.language").alias("language"),
    get_json_object(RAW_JSON_COL, "$.type").alias("type"),
    get_json_object(RAW_JSON_COL, "$.primary_location.source.display_name").alias("journal")

)

silver_df = silver_df \
    .filter(col("work_id").isNotNull()) \
    .filter(col("title").isNotNull()) \
    .filter(col("publication_year").isNotNull()) \
    .dropDuplicates(["work_id"]) \
    .withColumn("publication_date", to_date(col("publication_date")))


silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("publication_year") \
    .saveAsTable("openalex_lakehouse.silver.works")
    

silver_df.show()
silver_df.printSchema()
